In [0]:
%sql
drop table pyspark_scenarios_catalog.source.products_test1141

In [0]:
%sql
-- BY DEFAULT - DELTA
create table pyspark_scenarios_catalog.source.products_test1141
as select * from (
                   select *, max(updatedDate) over (partition by id) as max_update_date
                   from pyspark_scenarios_catalog.source.productsscd2src
                   ) cc where cc.updatedDate = cc.max_update_date

In [0]:
%sql
ALTER TABLE pyspark_scenarios_catalog.source.products_test1141 
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
df = spark.read\
    .table('pyspark_scenarios_catalog.source.products_test1141')
df.write.format('delta').mode('overwrite')\
    .save("/Volumes/pyspark_scenarios_catalog/source/db_volume/managed_tables_dir/products_test1141/sink/data11411")

In [0]:
%sql
truncate table delta.`/Volumes/pyspark_scenarios_catalog/source/db_volume/managed_tables_dir/products_test1141/sink/data11411`

In [0]:
"""
By default, Delta streaming assumes the source table is append-only.

But when you do a MERGE into another Delta table, you’re creating updates in the source Delta table’s transaction log.

The streaming reader detects these updates and errors out because it doesn’t know how to handle them (it only expects appends).

Enable Change Data Feed (best for UPSERT/SCD2)

Delta Lake supports Change Data Feed (CDF), which lets streaming readers consume updates/deletes/inserts.

Enable CDF on your source Delta table:
ALTER TABLE pyspark_scenarios_catalog.source.products_test1141 
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

.option("readChangeData", "true")
.option("startingVersion",  "latest")
"""

df = spark.readStream\
        .option("readChangeData", "true")\
        .option("startingVersion", 6)\
    .table('pyspark_scenarios_catalog.source.products_test1141')\
            .filter("_change_type IN ('insert', 'update_postimage')")

In [0]:
from delta.tables import DeltaTable

delta_tbl_obj = DeltaTable.forPath(spark, '/Volumes/pyspark_scenarios_catalog/source/db_volume/managed_tables_dir/products_test1141/sink/data11411')

def upsert_method(dfsrc, epochId):
    print("EPOCH ID = ")
    dfsrc.show(truncate=False)
    #display(dfsrc)
    #dfsrc.dropDuplicates(['id'])
    dfsrc.createOrReplaceTempView("my_temp_df")
    dfsrc = spark.sql("""select * from (
                   select *, max(updatedDate) over (partition by id) as max_update_date_col
                   from my_temp_df
                   ) cc where cc.updatedDate = cc.max_update_date_col""")
    delta_tbl_obj.alias('trg').merge(
        dfsrc.alias('src'),
        'trg.id = src.id'
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
# from pyspark.sql import functions as F
# from pyspark.sql.window import Window

# df = df.withColumn("max_update_date", F.max("updatedDate").over(Window.partitionBy("id")))\
#         .filter(F.col("updatedDate") == F.col("max_update_date"))\
#             .drop("max_update_date")   

In [0]:
df.writeStream.format('delta')\
    .outputMode('update')\
    .foreachBatch(upsert_method)\
    .option("checkpointLocation", "/Volumes/pyspark_scenarios_catalog/source/db_volume/managed_tables_dir/products_test1141/checkpoint11411")\
        .trigger(once=True)\
        .start()

In [0]:
%sql
select * from delta.`/Volumes/pyspark_scenarios_catalog/source/db_volume/managed_tables_dir/products_test1141/sink/data11411`

In [0]:
%sql
select * from pyspark_scenarios_catalog.source.products_test1141